# Tratativas

### Imports:

In [1]:
%pip install pandas
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

### Criação do banco de dados .csv

In [3]:
nome_arquivo_excel = '../dados/base_quedas_AACD_anonimizada.xlsx'
nome_arquivo_csv = '../dados/base_quedas_AACD_anonimizada.csv'

try:
    # Carregar as duas abas do Excel
    print("Lendo as abas do Excel...")
    df_60_mais = pd.read_excel(nome_arquivo_excel, sheet_name='60+')
    df_60_menos = pd.read_excel(nome_arquivo_excel, sheet_name='60-')

    # Função para juntar as linhas de 2 em 2
    def consertar_linhas_duplas(df):
        # Agrupa de 2 em 2 linhas e pega o primeiro valor não nulo
        df_agrupado = df.groupby(df.index // 2).first()
        return df_agrupado

    # Aplicar a correção nas duas tabelas
    print("Corrigindo o cadastro feito em duas linhas...")
    df_60_mais_corrigido = consertar_linhas_duplas(df_60_mais)
    df_60_menos_corrigido = consertar_linhas_duplas(df_60_menos)

    # Juntar as duas tabelas em uma só
    print("Unindo as duas tabelas...")
    df_final = pd.concat([df_60_mais_corrigido, df_60_menos_corrigido], ignore_index=True)

    # ---------------------------------------------------------
    print("Corrigindo a numeração dos IDs...")
    df_final['ID'] = range(1, len(df_final) + 1)
    # ---------------------------------------------------------

    # Salvar o resultado no arquivo .csv
    df_final.to_csv(nome_arquivo_csv, index=False, encoding='utf-8-sig')
    
    print(f"Sucesso! Arquivo '{nome_arquivo_csv}' criado com IDs sequenciais.")

except FileNotFoundError:
    print(f"Erro: O arquivo '{nome_arquivo_excel}' não foi encontrado na pasta.")
except Exception as e:
    print(f"Ocorreu um erro: {e}")

Lendo as abas do Excel...
Erro: O arquivo '../dados/base_quedas_AACD_anonimizada.xlsx' não foi encontrado na pasta.


### Correções do nome das colunas:

In [4]:
df = pd.read_csv('../dados/base_quedas_AACD_anonimizada.csv')

colunas = df.columns.tolist()
print(colunas)

['ID', 'Sexo', 'Idade', 'Clínica/Dx', 'Diagnóstico Funcional', 'Polifarmácia', 'Suporte social (terapêutico)', 'Suporte social (domiciliar)', 'Status funcional', 'Cognição (MOCA)', 'Cogniçao (10-CS)', 'Nº quedas ao chão', 'Queda último mês', 'Sentar/levantar 30s', 'Ponte', 'Dinamômetro', 'Panturrilha D', 'Panturrilha E', 'Braço D', 'Braço E', '10 metros', 'Aditamento 10m', 'CIF equilíbrio', 'Posição equilíbrio', 'Reação (blazepod)', 'Reação (blazepod) DT', 'Custo dupla tarefa']


In [5]:
df.rename(columns={'Polifarmácia?': 'Polifarmácia'}, inplace=True)
print(df.columns)

Index(['ID', 'Sexo', 'Idade', 'Clínica/Dx', 'Diagnóstico Funcional',
       'Polifarmácia', 'Suporte social (terapêutico)',
       'Suporte social (domiciliar)', 'Status funcional', 'Cognição (MOCA)',
       'Cogniçao (10-CS)', 'Nº quedas ao chão', 'Queda último mês',
       'Sentar/levantar 30s', 'Ponte', 'Dinamômetro', 'Panturrilha D',
       'Panturrilha E', 'Braço D', 'Braço E', '10 metros', 'Aditamento 10m',
       'CIF equilíbrio', 'Posição equilíbrio', 'Reação (blazepod)',
       'Reação (blazepod) DT', 'Custo dupla tarefa'],
      dtype='str')


C:\Users\49048057809\AppData\Local\Temp\ipykernel_11476\1449232662.py:1: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={'Polifarmácia?': 'Polifarmácia'}, inplace=True)


In [6]:
df.rename(columns={'Idade ': 'Idade'}, inplace=True)
print(df.columns)

Index(['ID', 'Sexo', 'Idade', 'Clínica/Dx', 'Diagnóstico Funcional',
       'Polifarmácia', 'Suporte social (terapêutico)',
       'Suporte social (domiciliar)', 'Status funcional', 'Cognição (MOCA)',
       'Cogniçao (10-CS)', 'Nº quedas ao chão', 'Queda último mês',
       'Sentar/levantar 30s', 'Ponte', 'Dinamômetro', 'Panturrilha D',
       'Panturrilha E', 'Braço D', 'Braço E', '10 metros', 'Aditamento 10m',
       'CIF equilíbrio', 'Posição equilíbrio', 'Reação (blazepod)',
       'Reação (blazepod) DT', 'Custo dupla tarefa'],
      dtype='str')


C:\Users\49048057809\AppData\Local\Temp\ipykernel_11476\2413175864.py:1: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={'Idade ': 'Idade'}, inplace=True)


### Correções nas colunas

- Apaga qualquer espaço em branco antes ou depois da palavra, transforma 'sim ' em 'sim'.

In [7]:
df['Polifarmácia'] = df['Polifarmácia'].replace(['nan', 'none', ''], 'não')

df['Polifarmácia'] = df['Polifarmácia'].astype('category')

print(df['Polifarmácia'].dtype)
print(df['Polifarmácia'].value_counts())

category
Polifarmácia
não    143
sim    139
Name: count, dtype: int64


- Remove "?" das idades e remove a letra "k" da posição 275 e substitui pela media

In [8]:
df['Idade'] = df['Idade'].astype(str).str.replace('?', '', regex=False)
df['Idade'] = df['Idade'].astype(str).str.replace('k', '', regex=False)

In [9]:
# errors='coerce' transforma valores inválidos/texto em NaN para não quebrar o código
df['Idade'] = pd.to_numeric(df['Idade'], errors='coerce')

media_idade = df['Idade'].mean()
media_idade = int(round(media_idade))

df.loc[df['ID'] == 275, 'Idade'] = media_idade

df['Idade'] = df['Idade'].fillna(media_idade).astype(int)


#### Correções de tipos nas colunas de numeros

In [10]:
df.tail(1)

,ID,Sexo,Idade,Clínica/Dx,Diagnóstico Funcional,Polifarmácia,Suporte social (terapêutico),Suporte social (domiciliar),Status funcional,Cognição (MOCA),...,Panturrilha E,Braço D,Braço E,10 metros,Aditamento 10m,CIF equilíbrio,Posição equilíbrio,Reação (blazepod),Reação (blazepod) DT,Custo dupla tarefa
281,282,M,27,AMP,NaN,não,namorada,NaN,marcha comunitária,NaN,...,NaN,NaN,NaN,7.0,Muletas canadenses,0.0,Ortostatismo,51.0,29.0,NaN


In [11]:
df.dtypes

ID                                 int64
Sexo                                 str
Idade                              int64
Clínica/Dx                           str
Diagnóstico Funcional                str
Polifarmácia                    category
Suporte social (terapêutico)         str
Suporte social (domiciliar)          str
Status funcional                     str
Cognição (MOCA)                  float64
Cogniçao (10-CS)                 float64
Nº quedas ao chão                float64
Queda último mês                     str
Sentar/levantar 30s              float64
Ponte                            float64
Dinamômetro                      float64
Panturrilha D                        str
Panturrilha E                    float64
Braço D                          float64
Braço E                          float64
10 metros                        float64
Aditamento 10m                       str
CIF equilíbrio                   float64
Posição equilíbrio                   str
Reação (blazepod

In [12]:
df = df.rename(columns={'Cognição         (10-CS)': 'Cogniçao (10-CS)'})

In [13]:
df['Reação (blazepod)'] = pd.to_numeric(df['Reação (blazepod)'], errors='coerce').astype('Int64')
df['Reação (blazepod) DT'] = pd.to_numeric(df['Reação (blazepod) DT'], errors='coerce').astype('Int64')
df['CIF equilíbrio'] = pd.to_numeric(df['CIF equilíbrio'], errors='coerce').astype('Int64')
df['Ponte'] = pd.to_numeric(df['Ponte'], errors='coerce').astype('Int64')
df['Sentar/levantar 30s'] = pd.to_numeric(df['Sentar/levantar 30s'], errors='coerce').astype('Int64')
df['Nº quedas ao chão'] = pd.to_numeric(df['Nº quedas ao chão'], errors='coerce').astype('Int64')
df['Cogniçao (10-CS)'] = pd.to_numeric(df['Cogniçao (10-CS)'], errors='coerce').astype('Int64')
df['Cognição (MOCA)'] = pd.to_numeric(df['Cognição (MOCA)'], errors='coerce').astype('Int64')

In [ ]:
df['Sexo'] = df['Sexo'].astype('category')
df['Posição equilíbrio'] = df['Posição equilíbrio'].astype('category')
df['Queda último mês'] = df['Queda último mês'].astype('boolean')
df['Panturrilha D'] = pd.to_numeric(df['Panturrilha D'], errors='coerce').astype('float')

TypeError: Need to pass bool-like values

In [17]:
df.isnull().sum()

ID                                0
Sexo                              0
Idade                             0
Clínica/Dx                        0
Diagnóstico Funcional           213
Polifarmácia                      0
Suporte social (terapêutico)      5
Suporte social (domiciliar)      18
Status funcional                 18
Cognição (MOCA)                 206
Cogniçao (10-CS)                216
Nº quedas ao chão                 8
Queda último mês                  7
Sentar/levantar 30s             170
Ponte                           126
Dinamômetro                       5
Panturrilha D                   107
Panturrilha E                   120
Braço D                         249
Braço E                         257
10 metros                        46
Aditamento 10m                   52
CIF equilíbrio                    7
Posição equilíbrio                6
Reação (blazepod)                 7
Reação (blazepod) DT             15
Custo dupla tarefa              282
dtype: int64

In [15]:
df.dtypes

ID                                 int64
Sexo                            category
Idade                              int64
Clínica/Dx                           str
Diagnóstico Funcional                str
Polifarmácia                    category
Suporte social (terapêutico)         str
Suporte social (domiciliar)          str
Status funcional                     str
Cognição (MOCA)                    Int64
Cogniçao (10-CS)                   Int64
Nº quedas ao chão                  Int64
Queda último mês                     str
Sentar/levantar 30s                Int64
Ponte                              Int64
Dinamômetro                      float64
Panturrilha D                        str
Panturrilha E                    float64
Braço D                          float64
Braço E                          float64
10 metros                        float64
Aditamento 10m                       str
CIF equilíbrio                     Int64
Posição equilíbrio                   str
Reação (blazepod

### Salva as alterações diretamente no arquivo .csv

Facilita a visualização diretamente no arquivo

In [16]:
#df.to_csv('../dados/base_quedas_AACD_anonimizada.csv', index=False, encoding='utf-8-sig')
#print("Arquivo salvo com sucesso!")